# SymFormer / TBX11K — Local Runbook**Drop this notebook on a Windows PC with an NVIDIA GPU, pick any Python kernel, and Run All.**It builds its own environment, proves the whole pipeline on synthetic data *before* downloadinganything large, then trains and evaluates the real thing.Replicates the detection method of *Revisiting Computer-Aided Tuberculosis Diagnosis*(Liu et al., TPAMI 2023; [arXiv:2307.02848](https://arxiv.org/abs/2307.02848)).---### What this needs from the machine| | ||---|---|| OS | Windows (Linux/macOS work too, minus the CUDA wheels) || GPU | NVIDIA, ≥ 6 GB VRAM. Without one it falls back to CPU and only the smoke test is practical || Disk | ~70 GB free — the raw archive is tens of GB and is deleted after extraction || Python | **any version**, see below || Network | for the wheels (~2.5 GB) and the dataset |### Why the kernel's Python version doesn't matterA notebook cannot install into one interpreter and then *become* it. So this one never tries: itbuilds a pinned `.venv` and runs every real step as a subprocess against it. **The kernel itselfonly ever uses the Python standard library.** If the PC has no Python 3.11, `uv` downloads astandalone one. No kernel restarts, no "now switch kernels" step.### The order is deliberateThe smoke test (Phase 4) runs on generated data and comes **before** the download. If Phase 4 isgreen, everything after it is data and patience — you never spend 40 minutes on a download only tofind out torch can't see your GPU.### Time budget| Phase | Cold machine | Re-run ||---|---|---|| 1–4 setup + smoke | ~10 min | ~1 min || 5–7 download + prepare | 1–3 h | skipped || 9–10 baseline + SymFormer | ~30 min | — || 11 stage-2 classifier | ~30 min | — || 12 ablation (opt-in) | overnight | — |

## Phase 0 — configurationEverything tunable lives here. Nothing below this cell needs editing.

In [ ]:
# ============================== CONFIG =====================================
# Training (paper stage-1 detection recipe, §3.4 / Table 8)
EPOCHS       = 24
BATCH_SIZE   = 8          # drop to 4 if you OOM on 8 GB, and halve LR with it
LR           = 0.005      # 0.01 is tuned for batch 16; linear-scale with the batch
SEED         = 0
STACK        = "torchvision"   # "torchvision" (default) or "mmdet" (the paper's own framework)

# Which phases to run
RUN_STAGE2    = True      # phase 11: the classification head + all-images evaluation
RUN_ABLATION  = False     # phase 12: 13 cells x seeds. Overnight job -- opt in deliberately.
ABLATION_SEEDS = [0]      # [0, 1, 2] is the experiment that can actually resolve the effect
SKIP_DOWNLOAD = False     # True if you already have the data prepared

# Paths (relative to the project root; all are gitignored)
DATA_RAW = "data/raw"           # extracted TBX11K -- deleted-able once prepared
DATA_512 = "data/tbx11k_512"    # the compact 512px dataset we actually train on
DUMMY    = "data/dummy512"      # synthetic smoke-test fixture

# Where to fetch the code if you copied only this .ipynb
REPO_URL = "https://github.com/Manas-Maahir/FOP"
BRANCH   = "main"
# ===========================================================================
print("config loaded")

## Phase 1 — preflightChecks the machine *before* anything is downloaded or installed, so failures are cheap and legible.A missing GPU is a warning, not an error: the pipeline still runs on CPU, just slowly.

In [ ]:
import os, platform, shutil, subprocess, sys, urllib.request
from pathlib import Path

def _hr(t=""): print("=" * 74); print(t) if t else None; print("=" * 74) if t else None

print("=" * 74); print("PREFLIGHT"); print("=" * 74)

print(f"  os          : {platform.system()} {platform.release()} ({platform.machine()})")
print(f"  kernel py   : {sys.version.split()[0]}   <- version is irrelevant, see the intro")
print(f"  cwd         : {Path.cwd()}")

# --- GPU -------------------------------------------------------------------
HAS_GPU, GPU_NAME, GPU_VRAM_GB = False, "none", 0.0
try:
    out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                          "--format=csv,noheader"], capture_output=True, text=True, timeout=30)
    if out.returncode == 0 and out.stdout.strip():
        first = out.stdout.strip().splitlines()[0]
        HAS_GPU, GPU_NAME = True, first.strip()
        for tok in first.split(","):
            if "MiB" in tok:
                GPU_VRAM_GB = float(tok.strip().split()[0]) / 1024
except Exception:
    pass
print(f"  gpu         : {GPU_NAME}")

# --- disk ------------------------------------------------------------------
free_gb = shutil.disk_usage(Path.cwd()).free / 1e9
print(f"  disk free   : {free_gb:.0f} GB")

# --- network ---------------------------------------------------------------
ONLINE = False
try:
    urllib.request.urlopen("https://pypi.org", timeout=15)
    ONLINE = True
except Exception:
    pass
print(f"  network     : {'ok' if ONLINE else 'UNREACHABLE'}")

# --- verdict ---------------------------------------------------------------
print()
problems, warnings = [], []
if not ONLINE:
    problems.append("No network. The wheels (~2.5 GB) and the dataset both need it.")
if free_gb < 70 and not SKIP_DOWNLOAD:
    warnings.append(f"Only {free_gb:.0f} GB free; the raw dataset needs ~70 GB of headroom. "
                    f"Phases 1-4 will still work -- they need under 10 GB.")
if not HAS_GPU:
    warnings.append("No NVIDIA GPU detected. CPU wheels will be installed: the smoke test still "
                    "passes, but a real 24-epoch run is impractical.")
elif GPU_VRAM_GB and GPU_VRAM_GB < 7:
    warnings.append(f"{GPU_VRAM_GB:.1f} GB of VRAM. If training OOMs, set BATCH_SIZE = 4 and "
                    f"LR = {LR/2:g} in the config cell.")

for w in warnings: print(f"  [warn] {w}")
for p in problems: print(f"  [FAIL] {p}")
if problems:
    raise SystemExit("Preflight failed -- fix the above and re-run this cell.")
print("  preflight OK" + ("  (with warnings)" if warnings else ""))

## Phase 2 — locate the projectIf you copied the whole repo, this finds it. If you copied **only this notebook**, it downloads thecode as a zip — no `git` required.

In [ ]:
import io, zipfile

def find_project(start: Path):
    """Walk up looking for the repo markers."""
    for cand in [start, *start.parents]:
        if (cand / "tools" / "train.py").is_file() and (cand / "symformer_tb" / "sas.py").is_file():
            return cand
    return None

PROJECT = find_project(Path.cwd())

if PROJECT is None:
    print("Project files not found next to this notebook -- fetching the repo ...")
    url = f"{REPO_URL.rstrip('/').removesuffix('.git')}/archive/refs/heads/{BRANCH}.zip"
    print(f"  {url}")
    dest = Path.cwd() / "_symformer_src"
    dest.mkdir(exist_ok=True)
    with urllib.request.urlopen(url, timeout=120) as r:
        with zipfile.ZipFile(io.BytesIO(r.read())) as zf:
            zf.extractall(dest)
    inner = [p for p in dest.iterdir() if p.is_dir()]
    PROJECT = find_project(inner[0]) if inner else None
    if PROJECT is None:
        raise SystemExit(f"Downloaded the repo but could not find tools/train.py under {dest}")
    print(f"  extracted -> {PROJECT}")

os.chdir(PROJECT)
print(f"\nproject root: {PROJECT}")
for p in ("tools/train.py", "tools/val.py", "symformer_tb/sas.py", "scripts/setup_env.py"):
    print(f"   {'ok ' if (PROJECT / p).is_file() else 'MISSING'}  {p}")

## Phase 3 — build the environmentCreates `.venv` on **Python 3.11** and installs a pinned stack. The version choice is forced, notarbitrary: OpenMMLab ships prebuilt `win_amd64` mmcv wheels (so no Visual Studio needed) only forcp38–cp311, and `mmdet 3.3.0` asserts `mmcv < 2.2.0`, whose Windows wheel is built against**torch 2.1.0**. Pinning both stacks to that one torch also removes a version confound from anytorchvision-vs-mmdet comparison.| | pin ||---|---|| Python | 3.11 (downloaded by `uv` if absent) || torch / torchvision | 2.1.0+cu121 / 0.16.0+cu121 || mmengine / mmcv / mmdet | 0.10.7 / 2.1.0 / 3.3.0 || numpy / pycocotools | <2 / 2.0.7 (mmcv is numpy-1.x ABI) |Stages are verified in order and **mmdet is last and optional** — if it fails, the torchvision stackis already complete and only `--stack mmdet` becomes unavailable. Re-running is nearly free: a stampfile records the pins and skips the work.> First run downloads ~2.5 GB. Expect 5–15 minutes.

In [ ]:
import json, subprocess, sys

r = subprocess.run([sys.executable, "scripts/setup_env.py"], cwd=PROJECT)
if r.returncode != 0:
    raise SystemExit("Environment setup failed -- read the output above. "
                     "Most common cause: no network or a proxy blocking download.pytorch.org")

VENV_PY = PROJECT / ".venv" / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
stamp = json.loads((PROJECT / ".venv" / ".symformer_stamp.json").read_text())
HAVE_MMDET = bool(stamp.get("have_mmdet"))
CUDA_OK    = bool(stamp.get("versions", {}).get("torch")) and stamp.get("gpu", "") != "nvidia-smi not found"

print("\n" + "=" * 74)
print(f"  venv python : {VENV_PY}")
print(f"  stacks      : torchvision{' + mmdet' if HAVE_MMDET else ''}")
if not HAVE_MMDET:
    print("  note        : mmdet unavailable; --stack mmdet cells will be skipped.")
print("=" * 74)

### The `run()` helperStreams subprocess output live so progress bars animate in the notebook instead of arriving as awall of text at the end.

In [ ]:
import codecs, subprocess, sys, time

def run(*cmd, check=True, quiet=False):
    """Run a command in the project venv, streaming its output as it happens."""
    cmd = [str(c) for c in cmd]
    if not quiet:
        print("$ " + " ".join(cmd[1:] if cmd[0] == str(VENV_PY) else cmd))
        sys.stdout.flush()
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONIOENCODING": "utf-8"}
    t0 = time.time()
    p = subprocess.Popen(cmd, cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         bufsize=0, env=env)
    dec = codecs.getincrementaldecoder("utf-8")("replace")
    while True:
        chunk = p.stdout.read1(16384)          # returns as soon as bytes are available, so \r works
        if not chunk:
            if p.poll() is not None:
                break
            continue
        sys.stdout.write(dec.decode(chunk))
        sys.stdout.flush()
    sys.stdout.write(dec.decode(b"", True))
    rc = p.wait()
    if not quiet:
        print(f"\n[exit {rc} in {time.time() - t0:.1f}s]")
    if check and rc != 0:
        raise SystemExit(f"Command failed (exit {rc}): {' '.join(cmd)}")
    return rc

def py(*args, **kw):
    """Run a project script inside the venv."""
    return run(VENV_PY, *args, **kw)

print("run() ready")

## Phase 4 — unit testsNo data needed. These cover the mirror/positional-encoding maths, the metric implementations, theadapters, checkpoint round-tripping, and the dataset's handling of images with no boxes.

In [ ]:
py("-m", "pytest", "tests/", "-q", "--no-header", check=False)
py("tools/prepare_tbx11k.py", "--selftest")

## Phase 5 — ⭐ SMOKE TEST (synthetic data, before any download)**This is the cell that decides whether the machine is set up correctly.**It generates a small synthetic dataset with the same layout and COCO schema as the real one, thendrives the entire chain against it — build the model, train, checkpoint, **kill and resume**,evaluate, write plots — in about a minute. It *asserts* rather than prints, so a broken installstops here instead of producing garbage three hours from now.If this cell is green, the remaining phases are just data and time.

In [ ]:
import csv, shutil

SMOKE = "runs/smoke"
shutil.rmtree(PROJECT / SMOKE, ignore_errors=True)

def check(label, ok, detail=""):
    print(f"  {'PASS' if ok else 'FAIL'}  {label}" + (f"   {detail}" if detail else ""))
    if not ok:
        raise SystemExit(f"SMOKE TEST FAILED: {label} {detail}")

def smoke_stack(stack, name):
    print("\n" + "=" * 74)
    print(f"SMOKE TEST -- stack: {stack}")
    print("=" * 74)
    run_dir = PROJECT / SMOKE / name

    # 1. two epochs on the synthetic set
    py("tools/train.py", "--data-root", DUMMY, "--stack", stack, "--no-sas",
       "--epochs", "2", "--batch-size", "2", "--image-size", "256",
       "--warmup-iters", "5", "--num-workers", "0",
       "--project", SMOKE, "--name", name, "--exist-ok")

    for artefact in ("results.csv", "args.yaml", "weights/last.pt", "weights/best.pt",
                     "results.png", "labels.jpg"):
        check(f"artefact {artefact}", (run_dir / artefact).is_file())

    rows = list(csv.DictReader(open(run_dir / "results.csv")))
    check("results.csv has 2 epochs", len(rows) == 2, f"got {len(rows)}")
    m = float(rows[-1].get("metrics/mAP50", "nan"))
    check("mAP50 is a finite number", m == m, f"mAP50={m}")
    loss = float(rows[-1]["train/total_loss"])
    check("loss is finite", loss == loss and abs(loss) < 1e6, f"loss={loss}")

    # 2. resume must CONTINUE, not restart
    py("tools/train.py", "--resume", f"{SMOKE}/{name}/weights/last.pt", "--epochs", "4")
    rows = list(csv.DictReader(open(run_dir / "results.csv")))
    check("resume continued to epoch 4", len(rows) == 4, f"got {len(rows)} rows")
    check("resume did not restart the epoch counter",
          [int(r["epoch"]) for r in rows] == [1, 2, 3, 4], [r["epoch"] for r in rows])

    # 3. standalone evaluation
    py("tools/val.py", "--weights", f"{SMOKE}/{name}/weights/best.pt",
       "--data-root", DUMMY, "--num-workers", "0")
    check("eval_log.jsonl written", (run_dir / "eval_log.jsonl").is_file())
    check("PR curve written", (run_dir / "PR_curve.png").is_file())
    print(f"\n  {stack}: all checks passed")

# generate the fixture once
py("tools/make_dummy_data.py", "--dst", DUMMY, "--size", "256",
   "--n-train", "16", "--n-val", "8", "--n-neg-train", "8", "--n-neg-val", "8")

smoke_stack("torchvision", "tv")
if HAVE_MMDET:
    smoke_stack("mmdet", "mmdet")
else:
    print("\n  SKIP mmdet smoke test -- mmdet is not installed (torchvision is unaffected)")

print("\n" + "=" * 74)
print("  SMOKE TEST GREEN -- the environment and the whole pipeline work.")
print("  Everything from here on is data and patience.")
print("=" * 74)

## Phase 6 — download the raw datasetTens of GB. The archive is deleted after extraction, so peak disk is roughly 2× the archive size.**Google Drive throttles large public files** and serves an HTML interstitial instead of the data.The script detects that and prints the manual route rather than failing silently — if it stops here,follow the printed instructions and re-run this cell. Nothing above needs redoing.Set `SKIP_DOWNLOAD = True` in the config cell if the data is already prepared.

In [ ]:
prepared = (PROJECT / DATA_512 / "annotations" / "tb_train_agnostic.json").is_file()

if SKIP_DOWNLOAD or prepared:
    print("compact dataset already present -- skipping download" if prepared
          else "SKIP_DOWNLOAD is set")
else:
    rc = py("scripts/download_tbx11k.py", "--dst", DATA_RAW, check=False)
    if rc != 0:
        print("\n" + "!" * 74)
        print("Download did not complete. Follow the instructions printed above, then re-run")
        print("this cell. Phases 1-5 do not need to be repeated.")
        print("!" * 74)

## Phase 7 — inspect the layout (gate)The official README documents the download links but **not** the archive's folder structure, so wediscover it rather than assume it.**Read the output before continuing.** You want: ~11,200 images, ~1,200 XML annotations, every XMLclass name mapped (no `!! UNMAPPED`), every image classified as healthy / sick / tb (no`!! UNCLASSIFIED`), and the split lists resolving to the `TBX11K_train.txt` / `TBX11K_val.txt`pair — **not** a `*_trainval.txt`, which would leak validation images into training and silentlyinflate every AP.

In [ ]:
if not prepared:
    py("tools/prepare_tbx11k.py", "--inspect", "--src", DATA_RAW)
else:
    print("dataset already prepared -- skipping inspection")

## Phase 8 — build the compact 512px datasetResizes all 11,200 images to 512×512 (scaling boxes by the same factors) and writes:| file | what it is ||---|---|| `tb_{train,val}_agnostic.json` | TB images only — **stage-1 detection training**, per paper §3.4 || `all_{train,val}_agnostic.json` | every image; non-TB carry zero boxes — the all-images eval mode || `cls_{train,val}.json` | image-level class labels — the stage-2 classifier |Decoding 11,200 3000×3000 PNGs is the bottleneck, so it runs across cores. It skips files alreadywritten, so an interrupted run resumes by re-running this cell.

In [ ]:
if not prepared:
    py("tools/prepare_tbx11k.py", "--src", DATA_RAW, "--dst", DATA_512,
       "--scope", "all", "--size", "512", "--write-agnostic")
    prepared = (PROJECT / DATA_512 / "annotations" / "tb_train_agnostic.json").is_file()

import json as _json
ann = PROJECT / DATA_512 / "annotations"
if prepared:
    print("\n" + "=" * 74); print("DATASET SUMMARY"); print("=" * 74)
    for f in sorted(ann.glob("*.json")):
        d = _json.loads(f.read_text())
        if "annotations" in d:
            print(f"  {f.name:32s} {len(d['images']):6d} images  {len(d['annotations']):6d} boxes")
        else:
            from collections import Counter
            c = Counter(r["class"] for r in d["images"])
            print(f"  {f.name:32s} {len(d['images']):6d} images  {dict(c)}")
    print("\n  Paper Table 2 expects: train 3000 healthy / 3000 sick / 600 TB")
    print("                         val    800 healthy /  800 sick / 200 TB")

In [ ]:
# Eyeball the data before training on it: box scale, aspect, and spatial density.
# The red line is the vertical centerline -- SymFormer's entire premise is that lesions break
# left-right symmetry, so how the boxes sit relative to it is worth a look.
from IPython.display import Image as _Img, display

fig = PROJECT / "runs" / "labels_preview.jpg"
if prepared:
    py("-c", (
        "import json,sys;sys.path.insert(0,'.');"
        "from symformer_tb.plotting import plot_labels;"
        f"d=json.load(open(r'{ann / 'tb_train_agnostic.json'}'));"
        "plot_labels([a['bbox'] for a in d['annotations']],512,"
        f"r'{fig}')"), quiet=True)
    display(_Img(filename=str(fig))) if fig.is_file() else print("(figure not produced)")

## Phase 9 — smoke test on the real dataOne epoch, five batches. Catches anything specific to the real images that the synthetic fixturecould not — wrong paths, unexpected image modes, box coordinates outside the frame.

In [ ]:
if prepared:
    shutil.rmtree(PROJECT / "runs/smoke/real", ignore_errors=True)
    py("tools/train.py", "--data-root", DATA_512, "--stack", STACK, "--no-sas",
       "--epochs", "1", "--batch-size", "2", "--limit-batches", "5",
       "--warmup-iters", "5", "--num-workers", "2",
       "--project", "runs/smoke", "--name", "real", "--exist-ok")
    print("\nReal-data smoke test done. Numbers here are meaningless (5 batches) -- "
          "what matters is that it ran.")

## Phase 10 — RetinaNet baselinePaper Table 8's first row: no attention, no positional encoding. This is the number SymFormer has tobeat.Reference: the Colab PoC's torchvision baseline landed at **79.1 AP50 / 33.4 AP**([report.md](../report.md) §5). Landing far from that means the port broke something.Interrupting is safe — Ctrl-C saves `last.pt`, and re-running the cell with `--resume` continues.

In [ ]:
if prepared:
    py("tools/train.py", "--data-root", DATA_512, "--stack", STACK, "--no-sas",
       "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE), "--lr", str(LR),
       "--seed", str(SEED), "--project", "runs/detect", "--name", "baseline", "--exist-ok")

## Phase 11 — SymFormer w/ RetinaNet (the primary comparison)The same model plus the shared **SAS** block: Symmetric Positional Encoding (with theidentity-initialised STN, right→left) and SymAttention.**The claim under test:** SymFormer's AP50 > baseline's. The paper reports 72.7 → 76.6 on its ownval ablation.

In [ ]:
if prepared:
    py("tools/train.py", "--data-root", DATA_512, "--stack", STACK,
       "--attention", "symattention", "--pe", "spe", "--stn", "--direction", "r2l",
       "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE), "--lr", str(LR),
       "--seed", str(SEED), "--project", "runs/detect", "--name", "symformer", "--exist-ok")

In [ ]:
# Side-by-side comparison of the two runs' final epochs.
import csv as _csv

def final_row(run):
    p = PROJECT / "runs/detect" / run / "results.csv"
    if not p.is_file():
        return None
    rows = list(_csv.DictReader(open(p)))
    return rows[-1] if rows else None

base, sym = final_row("baseline"), final_row("symformer")
if base and sym:
    print("=" * 74)
    print("PRIMARY COMPARISON  (final epoch -- best.pt is val-selected and biased)")
    print("=" * 74)
    print(f"{'model':<14}{'AP50':>10}{'AP':>10}{'P':>10}{'R':>10}")
    for name, r in (("baseline", base), ("SymFormer", sym)):
        print(f"{name:<14}{float(r['metrics/mAP50']):>10.1f}{float(r['metrics/mAP50-95']):>10.1f}"
              f"{float(r['metrics/precision'])*100:>10.1f}{float(r['metrics/recall'])*100:>10.1f}")
    d = float(sym["metrics/mAP50"]) - float(base["metrics/mAP50"])
    print(f"\n  delta AP50 = {d:+.1f}")
    print("  Paper's val ablation: 72.7 -> 76.6 (+3.9).")
    print("  One seed cannot resolve a difference this size -- run Phase 13 with 3 seeds")
    print("  before drawing any conclusion. See report.md section 6.")
else:
    print("Run phases 10 and 11 first.")

## Phase 12 — stage 2: the classification head + all-images evaluationThis is the half of SymFormer the Colab PoC never ran, because it needs the 10,000 non-TB imagesthat were out of scope there.Paper §3.4: freeze the backbone, FPN, SAS and detection head, then train only a 3-way classifier(healthy / sick-non-TB / TB) on **all** images for 12 epochs. At inference the classifier vetoesdetections on images it calls non-TB.Then detection is scored three ways, which is the point:| | ||---|---|| **TB-only** | the 200 TB val images — what the PoC reported || **all-images, unfiltered** | all 1,800 — false positives on healthy chests now count || **all-images, filtered** | same, with the classifier's veto applied |The gap between the last two *is* what stage 2 buys.

In [ ]:
if prepared and RUN_STAGE2:
    det = "runs/detect/symformer/weights/best.pt"
    if not (PROJECT / det).is_file():
        print(f"{det} not found -- run phase 11 first.")
    else:
        py("tools/train_cls.py", "--weights", det, "--data-root", DATA_512, "--epochs", "12")

In [ ]:
if prepared and RUN_STAGE2:
    det = "runs/detect/symformer/weights/best.pt"
    cls = "runs/classify/train/weights/best.pt"
    if (PROJECT / det).is_file():
        print("\n### TB-only mode")
        py("tools/val.py", "--weights", det, "--data-root", DATA_512,
           "--mode", "tb-only", "--tag", "symformer_tbonly")
        print("\n### All-images mode, no classifier")
        py("tools/val.py", "--weights", det, "--data-root", DATA_512,
           "--mode", "all", "--tag", "symformer_all_nofilter")
        if (PROJECT / cls).is_file():
            print("\n### All-images mode, classifier filtering false positives")
            py("tools/val.py", "--weights", det, "--data-root", DATA_512,
               "--mode", "all", "--cls-ckpt", cls, "--tag", "symformer_all_filtered")

## Phase 13 — Table 8 ablation (opt-in)13 configurations × your seeds. Set `RUN_ABLATION = True` and `ABLATION_SEEDS = [0, 1, 2]` in theconfig cell to run it.**Why seeds matter here.** [report.md](../report.md) §6 found the spread across six *different*configurations was ~2.3 AP50, while two runs of the *same* configuration differed by ~0.8. With oneseed there is no error bar and therefore no conclusion available in either direction. This is theproject's headline open question, and locally it costs only time.Budget ~10–15 min per cell per seed: 13 × 3 is an overnight job. It is resumable — re-run the celland finished cells are skipped.

In [ ]:
if prepared and RUN_ABLATION:
    py("tools/ablate.py", "--data-root", DATA_512, "--stack", STACK,
       "--seeds", *[str(s) for s in ABLATION_SEEDS],
       "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE), "--lr", str(LR),
       "--delete-weights", "--out", "results_ablation.md")
else:
    py("tools/ablate.py", "--data-root", DATA_512 or ".", "--dry-run", check=False)
    print("\nRUN_ABLATION is False -- set it to True in the config cell to actually run this.")

## Phase 14 — collect the resultsGathers every `eval_log.jsonl` under `runs/` into one table and writes `results_local.md`.

In [ ]:
import glob, json as _json

rows = []
for f in sorted(glob.glob(str(PROJECT / "runs" / "**" / "eval_log.jsonl"), recursive=True)):
    for line in Path(f).read_text().splitlines():
        line = line.strip()
        if line:
            try:
                rows.append(_json.loads(line))
            except _json.JSONDecodeError:
                pass

if not rows:
    print("No eval logs yet -- run at least phase 10.")
else:
    hdr = f"| {'run':<28} | {'mode':<8} | {'filt':<5} | {'AP50':>6} | {'AP':>6} | {'P':>6} | {'R':>6} |"
    sep = "|" + "|".join("-" * (len(c) + 2) for c in hdr.split("|")[1:-1]) + "|"
    out = ["# Local results", "",
           "Category-agnostic TB detection. `filt` = stage-2 classifier filtering applied.", "",
           hdr, sep]
    for r in rows:
        out.append(f"| {str(r.get('tag'))[:28]:<28} | {str(r.get('mode', '-')):<8} "
                   f"| {str(r.get('cls_filter', False)):<5} | {r.get('AP50', 0):>6.1f} "
                   f"| {r.get('AP', 0):>6.1f} | {r.get('precision', 0):>6.1f} "
                   f"| {r.get('recall', 0):>6.1f} |")
    text = "\n".join(out)
    (PROJECT / "results_local.md").write_text(text, encoding="utf-8")
    print(text)
    print(f"\nwritten -> {PROJECT / 'results_local.md'}")

---## Where everything went```runs/detect/baseline/          runs/detect/symformer/  weights/{best,last}.pt         args.yaml        what produced this run  results.csv                    results.png      loss + metric curves  labels.jpg                     PR_curve.png     precision vs recall  train_batch0.jpg               F1_curve.png     P/R/F1 vs confidence  val_batch0_pred.jpg            confusion_matrix.pngruns/classify/train/           stage-2 headruns/ablation/                 one dir per (cell, seed)results_local.md               the collected tableresults_ablation.md            Table 8 with mean ± std```## Reading the result honestlyThe Colab PoC produced a **null result**: every configuration landed within 77.7–80.0 AP50, a spreadno larger than the run-to-run noise of a single fixed configuration. That was not an implementationfailure — it was a power failure. ~600 training images and one seed cannot resolve a ~4-pointeffect. See [report.md](../report.md) §6.Nothing in this notebook changes that arithmetic by itself. Stage-1 detection still trains on TBimages only, because that is what the paper does. What has changed is what you can now do about it:1. **Phase 13 with 3+ seeds** — the actual fix, and now just an overnight run.2. **Phase 12's all-images mode** — a harder, more clinically honest metric the PoC never measured.3. **`STACK = "mmdet"`** — the paper's own anchors, losses and NMS, removing that confound.A difference between two configurations is only real if it exceeds the seed-to-seed spread in thesame row. With one seed, there is no spread and no claim.